<a href="https://colab.research.google.com/github/lucaslopez411-bot/Proyectos-Personales-Data-Science/blob/main/Analisis_Estadistico_Empresas_Chilenas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Análisis Estadístico de Indicadores Financieros en Empresas Chilenas

Proyecto de **estadística aplicada a información contable y financiera** sobre una muestra de empresas chilenas.

El análisis estudia especialmente la **rentabilidad empresarial y la distribución de dividendos**, combinando preparación de datos, construcción de ratios financieros, estadística descriptiva e inferencia estadística.

## Objetivos

- Evaluar la calidad y consistencia de información financiera empresarial.
- Construir indicadores de liquidez, endeudamiento, rotación, rentabilidad y efectivo.
- Analizar la distribución del ratio de rentabilidad.
- Comparar empresas según su política de distribución de dividendos.
- Aplicar pruebas de hipótesis para evaluar relaciones y diferencias estadísticamente significativas.

**Herramientas:** Python · Pandas · NumPy · SciPy · Statsmodels · Plotly · Matplotlib


## 1. Configuración del entorno
Se utilizan librerías de manipulación de datos, visualización y estadística inferencial. El notebook fue reorganizado para que pueda ejecutarse secuencialmente como proyecto reproducible.


In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import ttest_1samp, ttest_ind, f, chi2_contingency

pd.set_option("display.max_columns", None)


## 2. Carga y preparación del dataset

La base contiene información contable y financiera de empresas chilenas. Se convierten a formato numérico las variables necesarias para el análisis y se crean etiquetas descriptivas para sector y distribución de dividendos.


In [28]:
df_Chile = pd.read_csv("https://raw.githubusercontent.com/lucaslopez411-bot/Proyectos-Personales-Data-Science/refs/heads/main/Datasets/Base_Chile")


In [29]:
columnas_numericas = [
    "Activo.total", "Activo.Corriente", "Pasivo.total", "Pasivo.Corriente",
    "Efectivo", "Total.de.acciones", "Ventas", "Resultado.antes.impuest"
]

for col in columnas_numericas:
    df_Chile[col] = pd.to_numeric(df_Chile[col], errors="coerce")

print(f"Dimensiones iniciales: {df_Chile.shape[0]} filas × {df_Chile.shape[1]} columnas")
display(df_Chile.head())


Dimensiones iniciales: 1452 filas × 22 columnas


,Id,Sector,Dividendos,Activo.total,Activo.Corriente,Efectivo,Cuentas.por.cobrar.CP,Doc.por.Cob.Emp.Relc.CP,Pasivo.total,Pasivo.Corriente,Patrim.neto.consolidado,Capital.social,Ventas,Costo.de.Ventas,Resultado.Bruto,Result.operativo.,Resultado.antes.impuest,Ganancia.perdida.neta,Total.de.acciones,Reserva.Revalor.de.Cap,Reservas.de.Revaluacion,Reserva.Futuros.Divid
0,1,7,1,30394,1525.0,0.0,0.0,0,10891,10891,19503.0,19910,0,2590.0,-2590.0,-5540,-6303,-6260,25.0,NaN,NaN,NaN
1,2,7,1,30806,1680.0,0.0,0.0,0,17049,17049,13757.0,20467,0,1757.0,-1757.0,-4611,-4263,-4268,25.0,NaN,NaN,NaN
2,3,7,1,33007,2402.0,698.0,0.0,0,7527,7527,25480.0,19654,0,0.0,0.0,-833,-833,-833,25.0,NaN,NaN,NaN
3,4,8,1,288165,83834.0,0.0,0.0,83834,0,0,NaN,249180,0,0.0,0.0,-5931,-7823,-7823,3.0,0.0,0.0,0.0
4,5,8,0,316046,106134.0,23.0,0.0,0,38870,29811,NaN,289761,0,0.0,0.0,-102356,-11778,-11778,3.0,0.0,0.0,0.0


In [30]:
df_Chile['Dividendos_Categ'] = df_Chile['Dividendos'].map({
    0: 'No distribuyo',
    1: 'Distribuyo'})
df_Chile['Sector_Categ'] = df_Chile['Sector'].map({
    1: 'Comercial',
    2: 'Industrial',
    3: 'Indust y Comerc',
    4: 'Alimenticio',
    5: 'Construcción',
    6: 'Energía y Combust',
    7: 'Inmobiliaria',
    8: 'Serv y Telecom',
    9: 'Papelera y maderera',
    10: 'Otros'})

#Sirve para reemplazar los nombres de las variables númericas a categoricas


### Controles de consistencia financiera

Antes de calcular indicadores se verifican relaciones básicas entre partidas contables, por ejemplo que el activo y el total de acciones sean positivos y que las partidas corrientes no superen sus respectivos totales.


In [31]:
# Validaciones de consistencia financiera

validaciones = {
    'Activo total ≤ 0':
        (df_Chile['Activo.total'] <= 0),

    'Pasivo total < 0':
        (df_Chile['Pasivo.total'] < 0),

    'Pasivo corriente > Pasivo total':
        (df_Chile['Pasivo.Corriente'] > df_Chile['Pasivo.total']),

    'Activo corriente > Activo total':
        (df_Chile['Activo.Corriente'] > df_Chile['Activo.total']),

    'Efectivo > Activo total':
        (df_Chile['Efectivo'] > df_Chile['Activo.total']),

    'Total de acciones ≤ 0':
        (df_Chile['Total.de.acciones'] <= 0),

    'Ventas negativas':
        (df_Chile['Ventas'] < 0)
}

# Resumen
resultado_validaciones = pd.DataFrame({
    'Casos problemáticos': {
        nombre: condicion.sum()
        for nombre, condicion in validaciones.items()
    }
})

resultado_validaciones


,Casos problemáticos
Activo total ≤ 0,0
Pasivo total < 0,0
Pasivo corriente > Pasivo total,0
Activo corriente > Activo total,0
Efectivo > Activo total,0
Total de acciones ≤ 0,0
Ventas negativas,0


## 3. Construcción de ratios financieros

A partir de los estados contables se construyen cinco indicadores:

- **Liquidez:** Activo Corriente / Pasivo Corriente
- **Endeudamiento:** Pasivo Total / Activo Total
- **Rotación de activos:** Ventas / Activo Total
- **Rentabilidad:** Resultado antes de impuestos / Activo Total
- **Efectivo:** Efectivo / Activo Total


In [32]:
def crear_ratios(df_Chile):
    # Identificar las columnas necesarias para los ratios
    columnas_necesarias = ['Activo.Corriente', 'Pasivo.Corriente', 'Activo.total',
                          'Pasivo.total', 'Ventas', 'Resultado.antes.impuest', 'Efectivo']

    # Convertir estas columnas a numérico (coercer errores a NaN)
    for col in columnas_necesarias:
        if col in df_Chile.columns:
            df_Chile[col] = pd.to_numeric(df_Chile[col], errors='coerce')

    # Calcular los ratios
    df_Chile['Ratio_Liquidez'] = df_Chile['Activo.Corriente'] / df_Chile['Pasivo.Corriente']
    df_Chile['Ratio_Endeudamiento'] = df_Chile['Pasivo.total'] / df_Chile['Activo.total']
    df_Chile['Ratio_Rotacion_Activos'] = df_Chile['Ventas'] / df_Chile['Activo.total']
    df_Chile['Ratio_Rentabilidad'] = df_Chile['Resultado.antes.impuest'] / df_Chile['Activo.total']
    df_Chile['Ratio_Efectivo'] = df_Chile['Efectivo'] / df_Chile['Activo.total']

    return df_Chile

# Asegurarse de que pandas esté importado

df_Chile = crear_ratios(df_Chile)
df_Chile[['Ratio_Liquidez', 'Ratio_Endeudamiento', 'Ratio_Rotacion_Activos', 'Ratio_Rentabilidad', 'Ratio_Efectivo']].head()


,Ratio_Liquidez,Ratio_Endeudamiento,Ratio_Rotacion_Activos,Ratio_Rentabilidad,Ratio_Efectivo
0,0.140024,0.358327,0.0,-0.207376,0.000000
1,0.098540,0.553431,0.0,-0.138382,0.000000
2,0.319118,0.228043,0.0,-0.025237,0.021147
3,inf,0.000000,0.0,-0.027148,0.000000
4,3.560229,0.122988,0.0,-0.037267,0.000073


## 4. Calidad de datos

Se revisan valores faltantes, infinitos y registros duplicados. Los infinitos generados por divisiones se convierten en `NaN` para tratarlos de forma consistente.


In [33]:
def analisis_valores_nulos(df):

    valores_nulos = df.isna().sum().sort_values(ascending=False)
    valores_nulos[valores_nulos > 0]
    porcentaje_valores_nulos = (df.isna().sum() / df.shape[0]) * 100
    porcentaje_valores_nulos[porcentaje_valores_nulos > 0]

    valores_nulos_df = pd.DataFrame({'Valores_nulos': valores_nulos, 'Porcentaje_nulos': porcentaje_valores_nulos})
    valores_nulos_df

    nulos_df = valores_nulos_df.sort_values(by="Porcentaje_nulos", ascending=False)


    return nulos_df

nulos_df = analisis_valores_nulos(df_Chile)
nulos_df

#Permite verificar la cantidad y porcentaje de valores nulos contenidas en las variables.


,Valores_nulos,Porcentaje_nulos
Reservas.de.Revaluacion,1341,92.355372
Reserva.Futuros.Divid,1338,92.148760
Reserva.Revalor.de.Cap,1335,91.942149
Patrim.neto.consolidado,173,11.914601
Costo.de.Ventas,61,4.201102
Resultado.Bruto,61,4.201102
Cuentas.por.cobrar.CP,54,3.719008
Total.de.acciones,21,1.446281
Ratio_Efectivo,15,1.033058
Efectivo,15,1.033058


In [34]:
df_Chile["Ratio_Liquidez"] = df_Chile["Ratio_Liquidez"].replace([np.inf, -np.inf], np.nan)

duplicados = df_Chile.duplicated().sum()
df_Chile = df_Chile.drop_duplicates().copy()

print(f"Duplicados eliminados: {duplicados}")
print(f"Dimensiones después de la limpieza: {df_Chile.shape}")


Duplicados eliminados: 0
Dimensiones después de la limpieza: (1452, 29)


## 5. Análisis exploratorio

### Distribución de dividendos

La variable objetivo distingue entre empresas que distribuyeron y no distribuyeron dividendos. Esta segmentación será utilizada posteriormente para comparar el comportamiento de la rentabilidad.


In [35]:
# Conteo de categorías
dist_dividendos = (
    df_Chile['Dividendos_Categ']
    .value_counts(dropna=False)
    .reset_index()
)

dist_dividendos.columns = ['Dividendos_Categ', 'Cantidad']

# Calcular porcentajes para mostrar en el gráfico
dist_dividendos['Porcentaje'] = (dist_dividendos['Cantidad'] / dist_dividendos['Cantidad'].sum() * 100).round(1)

# Gráfico de torta
fig = px.pie(
    dist_dividendos,
    values='Cantidad',
    names='Dividendos_Categ',
    title='Distribución de empresas según distribución de dividendos',
    hole=0,  # Si pones 0.3 en lugar de 0, sería un donut (anillo)
    color_discrete_sequence=px.colors.qualitative.Set2  # Paleta de colores
)

# Personalizar el texto que se muestra en el gráfico
fig.update_traces(
    textposition='inside',  # 'inside' o 'outside'
    textinfo='percent+label',  # Muestra porcentaje y etiqueta
    insidetextorientation='horizontal'  # Orientación del texto
)

# Ajustar diseño general
fig.update_layout(
    showlegend=True,
    legend_title_text='Categoría',
    font=dict(size=12)
)

fig.show()


In [36]:
tabla1 = pd.DataFrame({
    'Frecuencia': df_Chile['Dividendos'].value_counts().sort_index(),
})

tabla1['Porcentaje (%)'] = (
    tabla1['Frecuencia']
    / tabla1['Frecuencia'].sum()
    * 100
).round(2)

tabla1.index = ['No distribuye', 'Distribuye']
tabla1.index.name = 'Distribución de dividendos'

# Agregar fila Total
tabla1.loc['Total'] = [
    tabla1['Frecuencia'].sum(),
    tabla1['Porcentaje (%)'].sum()
]

tabla1


,Frecuencia,Porcentaje (%)
Distribución de dividendos,,
No distribuye,137.0,9.44
Distribuye,1315.0,90.56
Total,1452.0,100.00


### Distribución y dispersión de la rentabilidad

Se analiza el `Ratio_Rentabilidad` mediante histogramas, boxplots y estadísticos descriptivos. Los valores extremos se conservan en esta etapa porque forman parte de la heterogeneidad financiera observada en la muestra.


In [37]:
def plot_ratio_distribucion(df_Chile, ratio, bins=30):

    # Mantener solo ratios existentes
    ratio = [r for r in ratio if r in df_Chile.columns]

    n = len(ratio)
    n_rows = (n + 2) // 3

    fig = make_subplots(
    rows=n_rows,
    cols=3,
    vertical_spacing=0.12,  # más espacio entre filas
    subplot_titles=[
        f"{r}<br>N={df_Chile[r].replace([np.inf, -np.inf], np.nan).dropna().shape[0]}"
        f"<br>P1-P99"
        for r in ratio
       ]
        )

    fig.update_layout(
    title=dict(
        text="Distribución de los Ratios Financieros",
        x=0.5
    ),
    template="plotly_white",
    height=350 * n_rows,
    width=1200,
    bargap=0.05,
    margin=dict(t=130)  # antes el valor por defecto era muy pequeño
    )

    for i, ratio in enumerate(ratio):

        row = i // 3 + 1
        col = i % 3 + 1

        # Limpiar infinitos y nulos
        datos = (
            df_Chile[ratio]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        # Percentiles para eliminar outliers visualmente
        p1 = datos.quantile(0.01)
        p99 = datos.quantile(0.99)

        datos_plot = datos[
            (datos >= p1) &
            (datos <= p99)
        ]

        # Histograma
        fig.add_trace(
            go.Histogram(
                x=datos_plot,
                nbinsx=bins,
                name=ratio,
                showlegend=False
            ),
            row=row,
            col=col
        )

        # Mediana calculada sobre TODOS los datos
        mediana = datos.median()

        fig.add_vline(
            x=mediana,
            line_dash="dash",
            line_width=2,
            annotation_text=f"Med: {mediana:.2f}",
            row=row,
            col=col
        )

        # Ajustar eje X al rango mostrado
        fig.update_xaxes(
            range=[p1, p99],
            row=row,
            col=col
        )

    fig.update_layout(
        title="Distribución de los Ratios Financieros",
        template="plotly_white",
        height=350 * n_rows,
        width=1200,
        bargap=0.05
    )

    fig.show()


In [38]:
ratio = ['Ratio_Rentabilidad']
plot_ratio_distribucion(df_Chile, ratio)


In [39]:
def plot_boxplots_generales(df, ratio, p_inf=0.01, p_sup=0.99):

    ratio = [r for r in ratio if r in df.columns]

    n = len(ratio)
    n_rows = (n + 2) // 3

    fig = make_subplots(
        rows=n_rows,
        cols=3,
        subplot_titles=[f"{ratio} (P1-P99)" for ratio in ratio]
    )

    for i, ratio in enumerate(ratio):

        row = i // 3 + 1
        col = i % 3 + 1

        datos = (
            df[ratio]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        # Recorte para visualización
        li = datos.quantile(p_inf)
        ls = datos.quantile(p_sup)

        datos_filtrados = datos[
            (datos >= li) &
            (datos <= ls)
        ]

        fig.add_trace(
            go.Box(
                y=datos_filtrados,
                name=ratio,
                boxpoints='outliers',
                showlegend=False
            ),
            row=row,
            col=col
        )

    fig.update_layout(
        title=f"Boxplots Generales de los Ratios Financieros (P{int(p_inf*100)}-P{int(p_sup*100)})",
        template="plotly_white",
        height=350 * n_rows,
        width=1200
    )

    fig.show()


In [40]:
plot_boxplots_generales(df_Chile, ratio)


In [41]:
def tabla_tendencia_dispersion(df, ratio):

    if isinstance(ratio, str):
        ratio = [ratio]

    resultados = []

    for r in ratio:

        if r not in df.columns:
            continue

        datos = (
            df[r]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        media = datos.mean()
        desv = datos.std()

        # Coeficiente de variación (%)
        cv = (desv / abs(media) * 100) if media != 0 else np.nan

        resultados.append({
            'Ratio': r,
            'N': len(datos),
            'Media': media,
            'Mediana': datos.median(),
            'Desv_Estandar': desv,
            'Varianza': datos.var(),
            'CV (%)': cv,
            'Min': datos.min(),
            'Max': datos.max(),
        })

    return pd.DataFrame(resultados).round(4)


In [42]:
display(tabla_tendencia_dispersion(df_Chile, ratio))


,Ratio,N,Media,Mediana,Desv_Estandar,Varianza,CV (%),Min,Max
0,Ratio_Rentabilidad,1452,0.0476,0.0372,0.1065,0.0113,223.5263,-1.7498,0.9123


In [43]:
def tabla_posicion_forma(df, ratio):

    # Permite pasar un string o una lista
    if isinstance(ratio, str):
        ratio = [ratio]

    resultados = []

    for r in ratio:

        if r not in df.columns:
            continue

        datos = (
            df[r]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        p1 = datos.quantile(0.01)
        q1 = datos.quantile(0.25)
        q2 = datos.quantile(0.50)
        q3 = datos.quantile(0.75)
        p99 = datos.quantile(0.99)

        iqr = q3 - q1

        skewness = datos.skew()

        resultados.append({
            'Ratio': r,
            'P1': p1,
            'Q1': q1,
            'Q2 (Mediana)': q2,
            'Q3': q3,
            'P99': p99,
            'IQR': iqr,
            'Asimetría': skewness,
        })

    return pd.DataFrame(resultados).round(4)


In [44]:
display(tabla_posicion_forma(df_Chile, ratio))


,Ratio,P1,Q1,Q2 (Mediana),Q3,P99,IQR,Asimetría
0,Ratio_Rentabilidad,-0.2101,0.0126,0.0372,0.0783,0.3078,0.0657,-5.2892


### Comparación según distribución de dividendos

Los boxplots permiten comparar visualmente la rentabilidad de ambos grupos. Se incluye también una visualización acotada al percentil 99 para facilitar la lectura sin eliminar definitivamente los valores extremos de la base.


In [45]:
def plot_boxplots_dividendos(df, ratio, variable_objetivo='Dividendos'):

    ratio = [r for r in ratio if r in df.columns]

    n = len(ratio)
    n_rows = (n + 2) // 3

    fig = make_subplots(
        rows=n_rows,
        cols=3,
        subplot_titles=ratio
    )

    # Diccionario para renombrar categorías
    etiquetas = {
        0: "No distribuyó",
        1: "Distribuyó"
    }

    for i, ratio in enumerate(ratio):

        row = i // 3 + 1
        col = i % 3 + 1

        for valor in sorted(df[variable_objetivo].dropna().unique()):

            datos = (
                df.loc[df[variable_objetivo] == valor, ratio]
                .replace([np.inf, -np.inf], np.nan)
                .dropna()
            )

            etiqueta = etiquetas.get(valor, str(valor))

            fig.add_trace(
                go.Box(
                    y=datos,
                    name=etiqueta,
                    boxpoints='outliers',
                    showlegend=(i == 0)  # Mostrar la leyenda una sola vez
                ),
                row=row,
                col=col
            )

    fig.update_layout(
        title="Ratios Financieros según Distribución de Dividendos",
        template="plotly_white",
        height=350 * n_rows,
        width=1200,
        legend_title="Dividendos"
    )

    fig.show()


In [46]:
plot_boxplots_dividendos(df_Chile, ratio, variable_objetivo='Dividendos')


In [47]:
def plot_boxplots_dividendos_p99(df,
                                 ratio,
                                 variable_objetivo='Dividendos'):

    ratio = [r for r in ratio if r in df.columns]

    etiquetas = {
        0: "No distribuyó dividendos",
        1: "Distribuyó dividendos"
    }

    n = len(ratio)
    n_rows = (n + 2) // 3

    fig = make_subplots(
        rows=n_rows,
        cols=3,
        subplot_titles=[f"{r} (P1–P99)" for r in ratio]
    )

    for i, ratio in enumerate(ratio):

        row = i // 3 + 1
        col = i % 3 + 1

        # Datos limpios del ratio
        datos_ratio = (
            df[ratio]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        # Percentiles
        p1 = datos_ratio.quantile(0.01)
        p99 = datos_ratio.quantile(0.99)

        for valor in sorted(df[variable_objetivo].dropna().unique()):

            datos = (
                df.loc[df[variable_objetivo] == valor, ratio]
                .replace([np.inf, -np.inf], np.nan)
                .dropna()
            )

            # Recorte visual
            datos = datos[
                (datos >= p1) &
                (datos <= p99)
            ]

            fig.add_trace(
                go.Box(
                    y=datos,
                    name=etiquetas.get(valor, str(valor)),
                    boxpoints='outliers',
                    showlegend=(i == 0)
                ),
                row=row,
                col=col
            )

    fig.update_layout(
        title="Ratios financieros según dividendos (Percentiles 1–99)",
        template="plotly_white",
        height=350 * n_rows,
        width=1200,
        legend_title="Dividendos"
    )

    fig.show()


In [48]:
plot_boxplots_dividendos_p99(df_Chile, ratio, variable_objetivo='Dividendos')


## 6. Tratamiento de valores faltantes

Para los ratios financieros se reemplazan valores infinitos por `NaN` y se imputan los faltantes mediante la **mediana**. Esta medida resulta apropiada para el análisis debido a la presencia de valores extremos y a la baja proporción de faltantes en los ratios considerados.


In [49]:
df_Chile_imputado = df_Chile.copy()

ratios = [
    'Ratio_Efectivo',
    'Ratio_Liquidez',
    'Ratio_Rotacion_Activos',
    'Ratio_Endeudamiento',
    'Ratio_Rentabilidad'
]

# Reemplazar infinitos por NaN
df_Chile_imputado[ratios] = df_Chile_imputado[ratios].replace(
    [np.inf, -np.inf],
    np.nan
)

# Imputar cada ratio con su propia mediana
for ratios in ratios:
    mediana = df_Chile_imputado[ratios].median()

    print(f"{ratios}: mediana utilizada = {mediana:.4f}")

    df_Chile_imputado[ratios] = (
        df_Chile_imputado[ratios]
        .fillna(mediana)
    )

# Verificación final
print("\nValores nulos luego de la imputación:")
print(df_Chile_imputado[ratios].isnull().sum())


Ratio_Efectivo: mediana utilizada = 0.0033
Ratio_Liquidez: mediana utilizada = 1.4100
Ratio_Rotacion_Activos: mediana utilizada = 0.1753
Ratio_Endeudamiento: mediana utilizada = 0.3181
Ratio_Rentabilidad: mediana utilizada = 0.0372

Valores nulos luego de la imputación:
0


In [50]:
ratios_financieros = [
    "Ratio_Liquidez",
    "Ratio_Endeudamiento",
    "Ratio_Rotacion_Activos",
    "Ratio_Rentabilidad",
    "Ratio_Efectivo"
]

print("Valores faltantes después de la imputación:")

display(
    df_Chile_imputado[ratios_financieros]
    .isna()
    .sum()
    .to_frame(name="Nulos")
)

Valores faltantes después de la imputación:


,Nulos
Ratio_Liquidez,0
Ratio_Endeudamiento,0
Ratio_Rotacion_Activos,0
Ratio_Rentabilidad,0
Ratio_Efectivo,0


## 7. Estadística descriptiva por política de dividendos

Se comparan medidas de posición, dispersión y forma entre empresas que distribuyen dividendos y aquellas que no lo hacen. Esto permite complementar la comparación visual antes de realizar inferencia estadística.


In [51]:
tabla_posicion = (
    df_Chile_imputado
    .groupby('Dividendos')[ratios]
    .agg(
        N='count',
        P1=lambda x: x.quantile(0.01),
        Q1=lambda x: x.quantile(0.25),
        Q2='median',
        Q3=lambda x: x.quantile(0.75),
        P99=lambda x: x.quantile(0.99),
        Asimetria='skew'
    )
)

# Calcular IQR
tabla_posicion['IQR'] = tabla_posicion['Q3'] - tabla_posicion['Q1']

# Reordenar columnas
tabla_posicion = tabla_posicion[
    ['N', 'P1', 'Q1', 'Q2', 'Q3', 'P99', 'IQR', 'Asimetria']
]

# Renombrar grupos
tabla_posicion.index = [
    'No distribuyó dividendos',
    'Distribuyó dividendos'
]

# Redondear
tabla_posicion = tabla_posicion.round(4)

tabla_posicion.style.format({
    'P1':'{:.4f}',
    'Q1':'{:.4f}',
    'Q2':'{:.4f}',
    'Q3':'{:.4f}',
    'P99':'{:.4f}',
    'IQR':'{:.4f}',
    'Asimetria':'{:.4f}'
})


,N,P1,Q1,Q2,Q3,P99,IQR,Asimetria
No distribuyó dividendos,137,-1.0030,0.0279,0.0534,0.0959,0.3301,0.0680,-5.2638
Distribuyó dividendos,1315,-0.2000,0.0119,0.0350,0.0745,0.3030,0.0626,-1.0769


In [52]:
tablas = []

for grupo in df_Chile_imputado['Dividendos_Categ'].dropna().unique():

    tabla = tabla_tendencia_dispersion(
        df_Chile_imputado[df_Chile_imputado['Dividendos_Categ'] == grupo],
        ratio
    )

    tabla['Grupo'] = grupo

    tablas.append(tabla)

comparacion = pd.concat(tablas, ignore_index=True)

comparacion = comparacion[
    ['Grupo'] + [c for c in comparacion.columns if c != 'Grupo']
]
comparacion


,Grupo,Ratio,N,Media,Mediana,Desv_Estandar,Varianza,CV (%),Min,Max
0,Distribuyo,Ratio_Rentabilidad,1315,0.0473,0.0350,0.0842,0.0071,178.0048,-0.8363,0.4804
1,No distribuyo,Ratio_Rentabilidad,137,0.0507,0.0534,0.2289,0.0524,451.5294,-1.7498,0.9123


# 8. Inferencia estadística

Se utiliza un nivel de significancia de **α = 0,05**.

Las pruebas buscan responder cinco preguntas:

1. ¿La proporción de empresas que distribuyen dividendos difiere del 90%?
2. ¿La rentabilidad media de las empresas es positiva?
3. ¿La rentabilidad media difiere entre empresas que distribuyen y no distribuyen dividendos?
4. ¿La variabilidad de la rentabilidad difiere entre ambos grupos?
5. ¿Existe asociación entre el nivel de rentabilidad y la distribución de dividendos?


### H1 — Proporción de empresas que distribuyen dividendos

**H₀:** p = 0,90  
**H₁:** p ≠ 0,90

Se construye un intervalo de confianza de Wilson al 95% para la proporción de empresas que distribuyen dividendos.


In [53]:
# Empresas que distribuyen dividendos
x = df_Chile_imputado["Dividendos"].sum()

# Total de empresas
n = len(df_Chile_imputado)

# Proporción muestral
proporcion = x / n

# Intervalo de confianza del 95% (se mantiene el nivel de confianza de la prueba)
ic_inf, ic_sup = proportion_confint(
    count=x, nobs=n, alpha=0.05, method="wilson"
)

# Hipótesis (Actualizado al 90%)
p0 = 0.90
alpha = 0.05

# Lógica para prueba bilateral
if p0 < ic_inf or p0 > ic_sup:
    decision = "Se rechaza H₀"
else:
    decision = "No se rechaza H₀"

# Tabla de resultados adaptada
tabla_ic = pd.DataFrame({
    "Empresas que distribuyen": [x],
    "Tamaño muestral": [n],
    "Proporción muestral": [round(proporcion, 4)],
    "Límite inferior IC 95%": [round(ic_inf, 4)],
    "Límite superior IC 95%": [round(ic_sup, 4)],
    "Proporción hipotética (p₀)": [p0],
    "Hipótesis nula": [f"p = {p0}"],
    "Hipótesis alternativa": [f"p ≠ {p0}"],
    "Decisión": [decision]
})

# Mensajes por consola
print("Variable: Distribución de Dividendos")
print(f"Hipótesis Nula (H₀): La proporción de empresas que distribuyen dividendos es igual a {p0:.0%}.")
print(f"Hipótesis Alternativa (H₁): La proporción de empresas que distribuyen dividendos es diferente de {p0:.0%}.")
print(f"Nivel de Significancia (α): {alpha}")

print(f"\nProporción muestral: {proporcion:.4f}")
print(f"Intervalo de Confianza 95%: [{ic_inf:.4f}, {ic_sup:.4f}]")

if p0 < ic_inf or p0 > ic_sup:
    print(
        f"\nDecisión: Se rechaza la hipótesis nula (H₀) porque el valor hipotético ({p0:.2f}) "
        f"se encuentra fuera del intervalo de confianza [{ic_inf:.4f}, {ic_sup:.4f}]."
    )
    print(
        f"Conclusión: Existe evidencia estadística suficiente para afirmar que "
        f"la proporción de empresas que distribuyen dividendos es diferente del {p0:.0%}."
    )
else:
    print(
        f"\nDecisión: No se rechaza la hipótesis nula (H₀) porque el valor hipotético ({p0:.2f}) "
        f"está contenido dentro del intervalo de confianza [{ic_inf:.4f}, {ic_sup:.4f}]."
    )
    print(
        f"Conclusión: No existe evidencia estadística suficiente para afirmar que "
        f"la proporción de empresas que distribuyen dividendos sea diferente del {p0:.0%}."
    )

display(tabla_ic)


Variable: Distribución de Dividendos
Hipótesis Nula (H₀): La proporción de empresas que distribuyen dividendos es igual a 90%.
Hipótesis Alternativa (H₁): La proporción de empresas que distribuyen dividendos es diferente de 90%.
Nivel de Significancia (α): 0.05

Proporción muestral: 0.9056
Intervalo de Confianza 95%: [0.8895, 0.9196]

Decisión: No se rechaza la hipótesis nula (H₀) porque el valor hipotético (0.90) está contenido dentro del intervalo de confianza [0.8895, 0.9196].
Conclusión: No existe evidencia estadística suficiente para afirmar que la proporción de empresas que distribuyen dividendos sea diferente del 90%.


,Empresas que distribuyen,Tamaño muestral,Proporción muestral,Límite inferior IC 95%,Límite superior IC 95%,Proporción hipotética (p₀),Hipótesis nula,Hipótesis alternativa,Decisión
0,1315,1452,0.9056,0.8895,0.9196,0.9,p = 0.9,p ≠ 0.9,No se rechaza H₀


**Resultado observado:** la proporción muestral es aproximadamente **90,56%** y el IC 95% es **[88,95%; 91,96%]**. Como 90% se encuentra dentro del intervalo, no se rechaza H₀.

En esta muestra, la evidencia no permite afirmar que la proporción de empresas que distribuyen dividendos sea distinta del 90%.


### H2 — Rentabilidad media positiva

**H₀:** μ ≤ 0  
**H₁:** μ > 0

Se aplica una prueba t unilateral de una muestra sobre el ratio de rentabilidad.


In [54]:
# Ratio de rentabilidad
muestra = df_Chile_imputado["Ratio_Rentabilidad"]

# Media poblacional hipotética
mu0 = 0

# Prueba t de una muestra
t_stat, p_bilateral = ttest_1samp(muestra, popmean=mu0)

# Convertir a p-valor unilateral
if t_stat > 0:
    p_valor = p_bilateral / 2
else:
    p_valor = 1 - (p_bilateral / 2)

# Estadísticos descriptivos
media_muestral = muestra.mean()
desv = muestra.std()
n = len(muestra)

# Nivel de significancia
alpha = 0.05

# Decisión
decision = (
    "Se rechaza H₀"
    if p_valor < alpha
    else "No se rechaza H₀"
)

# Tabla de resultados
tabla_hipotesis = pd.DataFrame({
    "Hipótesis nula": ["μ ≤ 0"],
    "Hipótesis alternativa": ["μ > 0"],
    "Media hipotética (μ₀)": [mu0],
    "Media muestral": [round(media_muestral, 4)],
    "Desv. estándar": [round(desv, 4)],
    "Tamaño muestral": [n],
    "Estadístico t": [round(t_stat, 4)],
    "Valor p (unilateral)": [round(p_valor, 6)],
    "Decisión": [decision]
})

print("Variable: Ratio de Rentabilidad")
print("Hipótesis Nula (H₀): El ratio de rentabilidad medio de las empresas chilenas es menor o igual a cero.")
print("Hipótesis Alternativa (H₁): El ratio de rentabilidad medio de las empresas chilenas es positivo.")
print(f"Nivel de Significancia (α): {alpha}")

if p_valor < alpha:

    print(
        f"\nDecisión: Se rechaza la hipótesis nula (H₀) porque el valor p ({p_valor:.4f}) "
        f"es menor que el nivel de significancia ({alpha})."
    )

    print(
        "Conclusión: Existe evidencia estadística suficiente para afirmar que "
        "el ratio de rentabilidad medio de las empresas chilenas es positivo."
    )

else:

    print(
        f"\nDecisión: No se rechaza la hipótesis nula (H₀) porque el valor p ({p_valor:.4f}) "
        f"es mayor o igual que el nivel de significancia ({alpha})."
    )

    print(
        "Conclusión: No existe evidencia estadística suficiente para afirmar que "
        "el ratio de rentabilidad medio de las empresas chilenas sea positivo."
    )

display(tabla_hipotesis)


Variable: Ratio de Rentabilidad
Hipótesis Nula (H₀): El ratio de rentabilidad medio de las empresas chilenas es menor o igual a cero.
Hipótesis Alternativa (H₁): El ratio de rentabilidad medio de las empresas chilenas es positivo.
Nivel de Significancia (α): 0.05

Decisión: Se rechaza la hipótesis nula (H₀) porque el valor p (0.0000) es menor que el nivel de significancia (0.05).
Conclusión: Existe evidencia estadística suficiente para afirmar que el ratio de rentabilidad medio de las empresas chilenas es positivo.


,Hipótesis nula,Hipótesis alternativa,Media hipotética (μ₀),Media muestral,Desv. estándar,Tamaño muestral,Estadístico t,Valor p (unilateral),Decisión
0,μ ≤ 0,μ > 0,0,0.0476,0.1065,1452,17.0473,0.0,Se rechaza H₀


**Resultado:** se rechaza H₀ al 5%. La muestra aporta evidencia estadística de que el ratio medio de rentabilidad es positivo.


### H3 — Diferencia de rentabilidad media según dividendos

**H₀:** μ₁ = μ₂  
**H₁:** μ₁ ≠ μ₂

Se utiliza la **prueba t de Welch**, que no exige igualdad de varianzas entre los grupos.


In [55]:
from scipy.stats import ttest_ind

# Grupos
grupo_div = df_Chile_imputado.loc[
    df_Chile_imputado["Dividendos"] == 1,
    "Ratio_Rentabilidad"
]

grupo_no_div = df_Chile_imputado.loc[
    df_Chile_imputado["Dividendos"] == 0,
    "Ratio_Rentabilidad"
]

# Prueba t de Welch
t_stat, p_value = ttest_ind(
    grupo_div,
    grupo_no_div,
    equal_var=False
)

decision = (
    "Se rechaza H₀"
    if p_value < 0.05
    else "No se rechaza H₀"
)

tabla_ttest = pd.DataFrame({
    "Hipótesis nula (H₀)": [
        "μ Dividendos = μ No Dividendos"
    ],
    "Hipótesis alternativa (H₁)": [
        "μ Dividendos ≠ μ No Dividendos"
    ],
    "Media Dividendos": [round(grupo_div.mean(), 4)],
    "Media No Dividendos": [round(grupo_no_div.mean(), 4)],
    "Estadístico t": [round(t_stat, 4)],
    "Valor p": [round(p_value, 6)],
    "Decisión": [decision]
})

print("Variable: Ratio de Rentabilidad")
print("Hipótesis Nula (H₀): La media del ratio de rentabilidad de las empresas que distribuyen dividendos es igual a la media del ratio de rentabilidad de las empresas que no distribuyen dividendos.")
print("Hipótesis Alternativa (H₁): La media del ratio de rentabilidad de las empresas que distribuyen dividendos es diferente a la media del ratio de rentabilidad de las empresas que no distribuyen dividendos.")
print(f"Nivel de Significancia (α): {alpha}")

if p_value < alpha:

    print(
        f"\nDecisión: Se rechaza la hipótesis nula (H₀) porque el valor p ({p_value:.6f}) "
        f"es menor que el nivel de significancia ({alpha})."
    )

    print(
        "Conclusión: Existe evidencia estadística suficiente para afirmar que "
        "la media del ratio de rentabilidad de las empresas que distribuyen dividendos "
        "es diferente a la media del ratio de rentabilidad de las empresas que no distribuyen dividendos."
    )

else:

    print(
        f"\nDecisión: No se rechaza la hipótesis nula (H₀) porque el valor p ({p_value:.6f}) "
        f"es mayor o igual que el nivel de significancia ({alpha})."
    )

    print(
        "Conclusión: No existe evidencia estadística suficiente para afirmar que "
        "la media del ratio de rentabilidad de las empresas que distribuyen dividendos "
        "es diferente a la media del ratio de rentabilidad de las empresas que no distribuyen dividendos. "
        "Por lo tanto, ambas medias pueden considerarse estadísticamente similares."
    )

display(tabla_ttest)


Variable: Ratio de Rentabilidad
Hipótesis Nula (H₀): La media del ratio de rentabilidad de las empresas que distribuyen dividendos es igual a la media del ratio de rentabilidad de las empresas que no distribuyen dividendos.
Hipótesis Alternativa (H₁): La media del ratio de rentabilidad de las empresas que distribuyen dividendos es diferente a la media del ratio de rentabilidad de las empresas que no distribuyen dividendos.
Nivel de Significancia (α): 0.05

Decisión: No se rechaza la hipótesis nula (H₀) porque el valor p (0.863729) es mayor o igual que el nivel de significancia (0.05).
Conclusión: No existe evidencia estadística suficiente para afirmar que la media del ratio de rentabilidad de las empresas que distribuyen dividendos es diferente a la media del ratio de rentabilidad de las empresas que no distribuyen dividendos. Por lo tanto, ambas medias pueden considerarse estadísticamente similares.


,Hipótesis nula (H₀),Hipótesis alternativa (H₁),Media Dividendos,Media No Dividendos,Estadístico t,Valor p,Decisión
0,μ Dividendos = μ No Dividendos,μ Dividendos ≠ μ No Dividendos,0.0473,0.0507,-0.1719,0.863729,No se rechaza H₀


**Resultado:** `p ≈ 0,864`. No se rechaza H₀.

La evidencia no muestra una diferencia estadísticamente significativa entre las medias de rentabilidad de las empresas que distribuyen dividendos y las que no distribuyen.


### H4 — Diferencia en la variabilidad de la rentabilidad

Se contrasta la igualdad de varianzas entre ambos grupos mediante una prueba F.

**H₀:** σ²₁ = σ²₂  
**H₁:** σ²₁ ≠ σ²₂


In [56]:
from scipy.stats import f
import numpy as np

# Varianzas muestrales
var_div = np.var(grupo_div, ddof=1)
var_no_div = np.var(grupo_no_div, ddof=1)

# Colocar la mayor arriba para obtener F >= 1
if var_div >= var_no_div:
    F = var_div / var_no_div
    gl1 = len(grupo_div) - 1
    gl2 = len(grupo_no_div) - 1
else:
    F = var_no_div / var_div
    gl1 = len(grupo_no_div) - 1
    gl2 = len(grupo_div) - 1

# p-valor bilateral
p_value = 2 * min(
    f.cdf(F, gl1, gl2),
    1 - f.cdf(F, gl1, gl2)
)

alpha = 0.05

decision = (
    "Se rechaza H₀"
    if p_value < alpha
    else "No se rechaza H₀"
)

tabla_f = pd.DataFrame({
    "Hipótesis nula (H₀)": [
        "σ² Dividendos = σ² No Dividendos"
    ],
    "Hipótesis alternativa (H₁)": [
        "σ² Dividendos ≠ σ² No Dividendos"
    ],
    "Varianza Dividendos": [round(var_div, 6)],
    "Varianza No Dividendos": [round(var_no_div, 6)],
    "Estadístico F": [round(F, 4)],
    "Valor p": [round(p_value, 6)],
    "Decisión": [decision]
})

print("Variable: Ratio de Rentabilidad")

print("Hipótesis Nula (H₀): La varianza del ratio de rentabilidad de las empresas que distribuyen dividendos es igual a la varianza de las empresas que no distribuyen dividendos.")

print("Hipótesis Alternativa (H₁): La varianza del ratio de rentabilidad de las empresas que distribuyen dividendos es diferente a la varianza de las empresas que no distribuyen dividendos.")

print(f"Nivel de Significancia (α): {alpha}")

print(f"Valor p: {p_value:.6f}")

if p_value < alpha:

    print(
        f"\nDecisión: Se rechaza la hipótesis nula (H₀) porque el valor p ({p_value:.6f}) "
        f"es menor que el nivel de significancia ({alpha})."
    )

    print(
        "Conclusión: Existe evidencia estadística suficiente para afirmar que "
        "la variabilidad del ratio de rentabilidad difiere entre las empresas "
        "que distribuyen dividendos y aquellas que no distribuyen dividendos."
    )

else:

    print(
        f"\nDecisión: No se rechaza la hipótesis nula (H₀) porque el valor p ({p_value:.6f}) "
        f"es mayor o igual que el nivel de significancia ({alpha})."
    )

    print(
        "Conclusión: No existe evidencia estadística suficiente para afirmar que "
        "la variabilidad del ratio de rentabilidad difiere entre las empresas "
        "que distribuyen dividendos y aquellas que no distribuyen dividendos."
    )

display(tabla_f)


Variable: Ratio de Rentabilidad
Hipótesis Nula (H₀): La varianza del ratio de rentabilidad de las empresas que distribuyen dividendos es igual a la varianza de las empresas que no distribuyen dividendos.
Hipótesis Alternativa (H₁): La varianza del ratio de rentabilidad de las empresas que distribuyen dividendos es diferente a la varianza de las empresas que no distribuyen dividendos.
Nivel de Significancia (α): 0.05
Valor p: 0.000000

Decisión: Se rechaza la hipótesis nula (H₀) porque el valor p (0.000000) es menor que el nivel de significancia (0.05).
Conclusión: Existe evidencia estadística suficiente para afirmar que la variabilidad del ratio de rentabilidad difiere entre las empresas que distribuyen dividendos y aquellas que no distribuyen dividendos.


,Hipótesis nula (H₀),Hipótesis alternativa (H₁),Varianza Dividendos,Varianza No Dividendos,Estadístico F,Valor p,Decisión
0,σ² Dividendos = σ² No Dividendos,σ² Dividendos ≠ σ² No Dividendos,0.007092,0.052396,7.3885,0.0,Se rechaza H₀


**Resultado:** se rechaza H₀. Aunque las medias de rentabilidad no presentan diferencias estadísticamente significativas, la **dispersión de la rentabilidad sí difiere entre los grupos**.

Este resultado es relevante porque muestra que comparar únicamente promedios puede ocultar diferencias en el perfil de variabilidad financiera.


### H5 — Asociación entre nivel de rentabilidad y dividendos

Para estudiar la asociación, la rentabilidad se divide en cuartiles y se construye una tabla de contingencia. Luego se aplica una prueba **Chi-cuadrado de independencia**.

**H₀:** distribución de dividendos y nivel de rentabilidad son independientes.  
**H₁:** existe asociación entre ambas variables.


In [57]:
df_Chile_imputado["Rentabilidad_Cuartil"] = pd.qcut(
    df_Chile_imputado["Ratio_Rentabilidad"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

tabla_cont = pd.crosstab(
    df_Chile_imputado["Rentabilidad_Cuartil"],
    df_Chile_imputado["Dividendos_Categ"]
)

display(tabla_cont.T)


Rentabilidad_Cuartil,Q1,Q2,Q3,Q4
Dividendos_Categ,,,,
Distribuyo,338,345,319,313
No distribuyo,25,18,44,50


In [58]:
# Prueba Chi-cuadrado
chi2, p_value, gl, esperados = chi2_contingency(tabla_cont)

alpha = 0.05

decision = (
    "Se rechaza H₀"
    if p_value < alpha
    else "No se rechaza H₀"
)

tabla_chi2 = pd.DataFrame({
    "Hipótesis nula (H₀)": [
        "Dividendos independiente de Rentabilidad"
    ],
    "Hipótesis alternativa (H₁)": [
        "Dividendos depende de Rentabilidad"
    ],
    "Estadístico χ²": [round(chi2, 4)],
    "Grados de libertad": [gl],
    "Valor p": [round(p_value, 6)],
    "Decisión": [decision]
})

print("Variable: Distribución de Dividendos y Nivel de Rentabilidad")

print("Hipótesis Nula (H₀): La distribución de dividendos es independiente del nivel de rentabilidad.")

print("Hipótesis Alternativa (H₁): La distribución de dividendos depende del nivel de rentabilidad.")

print(f"Nivel de Significancia (α): {alpha}")

print(f"\nEstadístico χ²: {chi2:.4f}")
print(f"Grados de libertad: {gl}")
print(f"Valor p: {p_value:.6f}")

if p_value < alpha:

    print(
        f"\nDecisión: Se rechaza la hipótesis nula (H₀) porque el valor p ({p_value:.6f}) "
        f"es menor que el nivel de significancia ({alpha})."
    )

    print(
        "Conclusión: Existe evidencia estadística suficiente para afirmar que "
        "la distribución de dividendos depende del nivel de rentabilidad de las empresas."
    )

else:

    print(
        f"\nDecisión: No se rechaza la hipótesis nula (H₀) porque el valor p ({p_value:.6f}) "
        f"es mayor o igual que el nivel de significancia ({alpha})."
    )

    print(
        "Conclusión: No existe evidencia estadística suficiente para afirmar que "
        "la distribución de dividendos depende del nivel de rentabilidad."
    )

display(tabla_chi2)


Variable: Distribución de Dividendos y Nivel de Rentabilidad
Hipótesis Nula (H₀): La distribución de dividendos es independiente del nivel de rentabilidad.
Hipótesis Alternativa (H₁): La distribución de dividendos depende del nivel de rentabilidad.
Nivel de Significancia (α): 0.05

Estadístico χ²: 22.3335
Grados de libertad: 3
Valor p: 0.000056

Decisión: Se rechaza la hipótesis nula (H₀) porque el valor p (0.000056) es menor que el nivel de significancia (0.05).
Conclusión: Existe evidencia estadística suficiente para afirmar que la distribución de dividendos depende del nivel de rentabilidad de las empresas.


,Hipótesis nula (H₀),Hipótesis alternativa (H₁),Estadístico χ²,Grados de libertad,Valor p,Decisión
0,Dividendos independiente de Rentabilidad,Dividendos depende de Rentabilidad,22.3335,3,0.000056,Se rechaza H₀


**Resultado:** χ² ≈ **22,33**, con `p ≈ 0,000056`. Se rechaza H₀.

Existe evidencia estadística de asociación entre el nivel de rentabilidad —representado por cuartiles— y la distribución de dividendos. Este resultado no implica causalidad, pero sí indica que ambas variables no se comportan como independientes dentro de la muestra.


In [59]:
tabla_plot = tabla_cont.reset_index().melt(
    id_vars="Rentabilidad_Cuartil",
    var_name="Dividendos",
    value_name="Frecuencia"
)

tabla_plot["Porcentaje"] = (
    tabla_plot.groupby("Rentabilidad_Cuartil", observed=False)["Frecuencia"]
    .transform(lambda x: x / x.sum() * 100)
)

fig = px.bar(
    tabla_plot,
    x="Rentabilidad_Cuartil",
    y="Porcentaje",
    color="Dividendos",
    text=tabla_plot["Porcentaje"].round(1).astype(str) + "%"
)

fig.update_layout(
    title="Distribución de dividendos según cuartiles de rentabilidad",
    xaxis_title="Cuartil del ratio de rentabilidad",
    yaxis_title="Porcentaje de empresas",
    yaxis_ticksuffix="%",
    legend_title="Dividendos",
    template="plotly_white",
    height=550,
    width=900
)

fig.update_traces(textposition="inside")
fig.show()


# 9. Conclusiones

El análisis permite integrar información contable con herramientas de estadística descriptiva e inferencial.

### Principales hallazgos

- Aproximadamente **90,6%** de las empresas de la muestra distribuyen dividendos.
- Existe evidencia estadística de que la **rentabilidad media es positiva**.
- No se detecta una diferencia significativa en la **rentabilidad media** entre empresas que distribuyen y no distribuyen dividendos.
- Sí se observan diferencias estadísticamente significativas en la **variabilidad de la rentabilidad** entre ambos grupos.
- La prueba Chi-cuadrado muestra una **asociación entre los cuartiles de rentabilidad y la distribución de dividendos**.

### Interpretación

Los resultados muestran por qué el análisis financiero no debería limitarse a comparar promedios. Dos grupos pueden presentar rentabilidades medias similares y, al mismo tiempo, exhibir diferencias importantes en dispersión y en la forma en que la política de dividendos se relaciona con distintos niveles de rentabilidad.

### Posibles extensiones

- Incorporar el **sector económico** como variable explicativa.
- Evaluar simultáneamente liquidez, endeudamiento, rotación y rentabilidad.
- Aplicar modelos de regresión para estudiar los determinantes de la distribución de dividendos.
- Extender el análisis a diferentes períodos o países para evaluar estabilidad temporal y comparabilidad.
